# **Neural Reranking - FIXED Document Fetching**

## Problem Identified:
- `json.loads(doc.raw())` is failing
- Need to use `doc.lucene_document()` instead

This should make neural reranking actually work!

In [ ]:
# Setup (run the Java 21 + packages cells first if needed)
from google.colab import drive
import pandas as pd
import numpy as np
from tqdm import tqdm
from pyserini.search.lucene import LuceneSearcher
from trectools import TrecQrel, TrecRun, TrecEval
from tabulate import tabulate
from sentence_transformers import CrossEncoder

drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/TEXT/FinalProject'
QUERIES_PATH = BASE_PATH + '/queriesROBUST.txt'
QRELS_PATH = BASE_PATH + '/qrels_50_Queries'

queries_df = pd.read_csv(QUERIES_PATH, sep='\t', header=None, names=['qid', 'text'], dtype={'qid': str})
TRAIN_QIDS = sorted(queries_df['qid'].tolist(), key=int)[:50]
QUERIES_DICT = dict(zip(queries_df.qid, queries_df.text))

searcher = LuceneSearcher.from_prebuilt_index('robust04')
searcher.set_bm25(k1=0.6, b=0.4)

qrels = TrecQrel(QRELS_PATH)

print(f"✅ Loaded {len(TRAIN_QIDS)} queries")

## Test: Fixed Document Fetching

In [ ]:
# Test on first query
test_qid = TRAIN_QIDS[0]
test_query = QUERIES_DICT[test_qid]

print(f"Test Query: {test_query}\n")

hits = searcher.search(test_query, k=5)

print("Testing different document fetching methods:\n")

for i, hit in enumerate(hits, 1):
    print(f"{i}. DocID: {hit.docid}")
    
    doc = searcher.doc(hit.docid)
    
    # Method 1: doc.raw() - THIS WAS FAILING
    try:
        import json
        content1 = json.loads(doc.raw()).get('contents', '')
        print(f"   ✅ Method 1 (raw): {len(content1)} chars")
    except Exception as e:
        print(f"   ❌ Method 1 (raw): {e}")
    
    # Method 2: doc.lucene_document() - BETTER
    try:
        lucene_doc = doc.lucene_document()
        content2 = lucene_doc.get('contents')
        if content2:
            print(f"   ✅ Method 2 (lucene): {len(content2)} chars")
        else:
            print(f"   ⚠️ Method 2 (lucene): No 'contents' field")
    except Exception as e:
        print(f"   ❌ Method 2 (lucene): {e}")
    
    # Method 3: doc.contents() - SIMPLEST
    try:
        content3 = doc.contents()
        print(f"   ✅ Method 3 (contents): {len(content3)} chars")
    except Exception as e:
        print(f"   ❌ Method 3 (contents): {e}")
    
    print()

## Fixed Neural Reranking Function

In [ ]:
# Load cross-encoder
print("Loading cross-encoder...")
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', max_length=512)
print("✅ Model loaded")

def neural_rerank_FIXED(query, initial_scores, top_k=100):
    """
    FIXED: Use doc.contents() instead of json.loads(doc.raw())
    """
    # Get top-K by initial score
    top_docs = sorted(initial_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
    
    # Fetch document content - FIXED METHOD
    pairs = []
    valid_docids = []
    
    for docid, _ in top_docs:
        try:
            doc = searcher.doc(docid)
            if doc:
                # FIXED: Use doc.contents() directly
                content = doc.contents()
                if content:
                    # Take first 512 chars
                    pairs.append([query, content[:512]])
                    valid_docids.append(docid)
        except Exception as e:
            continue
    
    if not pairs:
        return initial_scores
    
    # Get neural scores
    raw_neural_scores = cross_encoder.predict(pairs, batch_size=32, show_progress_bar=False)
    
    # Scale neural scores to be ABOVE BM25 range
    max_bm25 = max(initial_scores.values())
    
    # Normalize neural scores
    min_neural = min(raw_neural_scores)
    max_neural = max(raw_neural_scores)
    neural_range = max_neural - min_neural
    
    if neural_range > 0:
        normalized = [(s - min_neural) / neural_range for s in raw_neural_scores]
    else:
        normalized = [0.5] * len(raw_neural_scores)
    
    # Scale: top neural doc = max_bm25 * 2, bottom = max_bm25 * 1.1
    scaled_scores = [max_bm25 * (1.1 + 0.9 * norm) for norm in normalized]
    
    # Build result
    result = initial_scores.copy()
    for docid, score in zip(valid_docids, scaled_scores):
        result[docid] = score
    
    return result

print("✅ Neural reranking function ready")

## Test Fixed Neural Reranking

In [ ]:
print("Testing FIXED neural reranking on sample query...\n")

# Get BM25 results
bm25_hits = searcher.search(test_query, k=100)
bm25_scores = {h.docid: float(h.score) for h in bm25_hits}

print(f"BM25 retrieved {len(bm25_scores)} documents")
print(f"BM25 score range: {min(bm25_scores.values()):.2f} to {max(bm25_scores.values()):.2f}\n")

# Apply neural reranking
print("Applying neural reranking...")
reranked = neural_rerank_FIXED(test_query, bm25_scores, top_k=100)

print(f"\nNeural reranked {len(reranked)} documents")

# Check if it actually changed the ranking
bm25_top10 = [docid for docid, _ in sorted(bm25_scores.items(), key=lambda x: x[1], reverse=True)[:10]]
neural_top10 = [docid for docid, _ in sorted(reranked.items(), key=lambda x: x[1], reverse=True)[:10]]

print("\n" + "="*80)
print("COMPARISON: BM25 vs Neural Top-10")
print("="*80)

comparison = []
for i in range(10):
    same = "✓" if bm25_top10[i] == neural_top10[i] else "✗"
    comparison.append([i+1, bm25_top10[i], neural_top10[i], same])

print(tabulate(comparison, headers=["Rank", "BM25 Doc", "Neural Doc", "Same?"], tablefmt="fancy_grid"))

same_count = sum(1 for row in comparison if row[3] == "✓")
print(f"\n{same_count}/10 documents unchanged")
print(f"{10-same_count}/10 documents reranked")

if same_count < 10:
    print("\n✅ SUCCESS! Neural reranking IS changing the ranking!")
else:
    print("\n⚠️ WARNING: Neural produces same ranking as BM25")

## Full Evaluation with Fixed Neural Reranking

In [ ]:
def get_bm25_scores(query, k=1000):
    hits = searcher.search(query, k=k)
    return {h.docid: float(h.score) for h in hits}

def get_rm3_scores(query, k=1000):
    searcher.set_rm3(fb_docs=10, fb_terms=10, original_query_weight=0.5)
    hits = searcher.search(query, k=k)
    searcher.unset_rm3()
    return {h.docid: float(h.score) for h in hits}

def evaluate(pipeline_func, name):
    all_rows = []
    for qid, query in tqdm(QUERIES_DICT.items(), desc=name):
        try:
            scores = pipeline_func(query)
            ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:1000]
            for rank, (docid, score) in enumerate(ranked, start=1):
                all_rows.append([qid, "Q0", docid, rank, score, name])
        except Exception as e:
            print(f"Error on {qid}: {e}")
    
    run_file = f"{name}_run.txt"
    pd.DataFrame(all_rows).to_csv(run_file, sep=' ', header=False, index=False)
    return TrecEval(TrecRun(run_file), qrels)

print("\n" + "="*80)
print("EVALUATION WITH FIXED NEURAL RERANKING")
print("="*80)

results = []

# Baseline
print("\n1. Baseline BM25")
te1 = evaluate(lambda q: get_bm25_scores(q), "1_Baseline_BM25")
results.append(["Baseline BM25", te1.get_map(), te1.get_precision(5), te1.get_precision(10)])

# RM3
print("\n2. BM25 + RM3")
te2 = evaluate(lambda q: get_rm3_scores(q), "2_BM25_RM3")
results.append(["BM25 + RM3", te2.get_map(), te2.get_precision(5), te2.get_precision(10)])

# Neural (FIXED)
print("\n3. BM25 + Neural (FIXED)")
te3 = evaluate(lambda q: neural_rerank_FIXED(q, get_bm25_scores(q), 100), "3_BM25_Neural_FIXED")
results.append(["BM25 + Neural (FIXED)", te3.get_map(), te3.get_precision(5), te3.get_precision(10)])

# RM3 + Neural (FIXED)
print("\n4. RM3 + Neural (FIXED)")
te4 = evaluate(lambda q: neural_rerank_FIXED(q, get_rm3_scores(q), 100), "4_RM3_Neural_FIXED")
results.append(["RM3 + Neural (FIXED)", te4.get_map(), te4.get_precision(5), te4.get_precision(10)])

# Display results
print("\n" + "="*80)
print("RESULTS WITH FIXED NEURAL RERANKING")
print("="*80)
print(tabulate(results, headers=["Pipeline", "MAP", "P@5", "P@10"], tablefmt="fancy_grid", floatfmt=".4f"))

baseline_map = results[0][1]
improvements = [[r[0], f"{((r[1] - baseline_map) / baseline_map * 100):+.2f}%"] for r in results[1:]]

print("\n" + "="*80)
print("IMPROVEMENT OVER BASELINE")
print("="*80)
print(tabulate(improvements, headers=["Pipeline", "MAP Gain"], tablefmt="fancy_grid"))

best_idx = max(range(len(results)), key=lambda i: results[i][1])
print(f"\n🏆 BEST: {results[best_idx][0]}")
print(f"   MAP: {results[best_idx][1]:.4f} ({((results[best_idx][1] - baseline_map) / baseline_map * 100):+.2f}%)")
print("="*80)